### RAG Pipelines - Data Ingestion to Vector DB Pipeline

In [14]:
# function to read all data files (pdf) and convert them into data document structure
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\Sanaullah\AppData\Local\Temp\ipykernel_15124\97167243.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [15]:
# read all the pdfs inside the directory
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: EE Tender.pdf


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/KFRBGO+MyriadPro-Regular', '/Encoding': IndirectObject(131, 0, 2350417500496), '/FirstChar': 30, '/FontDescriptor': IndirectObject(132, 0, 2350417500496), '/LastChar': 146, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [523, 815, 212, 0, 0, 0, 0, 0, 605, 0, 284, 284, 0, 0, 207, 307, 207, 343, 513, 513, 513, 513, 513, 513, 513, 513, 513, 513, 207, 0, 0, 0, 0, 0, 0, 612, 542, 580, 666, 492, 487, 646, 652, 239, 370, 542, 472, 804, 658, 689, 532, 689, 538, 493, 497, 647, 558, 846, 571, 541, 553, 0, 0, 0, 0, 0, 0, 482, 569, 448, 564, 501, 292, 559, 555, 234, 243, 469, 236, 834, 555, 549, 569, 563, 327, 396, 331, 551, 481, 736, 463, 471, 428, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 207]}, but is not installed. Consider installing fontTools if you encounter encoding problems.


  ✓ Loaded 2 pages

Processing: Transit-Map-Islamabad-V-06.pdf
  ✓ Loaded 1 pages

Processing: trizetto-email-thread.pdf
  ✓ Loaded 37 pages

Total documents loaded: 40


In [16]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-03-24T13:22:45+05:00', 'author': 'Dell', 'moddate': '2026-03-26T14:52:52+05:00', 'source': '..\\data\\pdf\\EE Tender.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'EE Tender.pdf', 'file_type': 'pdf'}, page_content="OFFICE OF THE ASSISTANT DIRECTOR LOCAL GOVERNMENT AND RURAL \nDEVELOPMENT DEPARTMENT MANSEHRA \nNOTICE INVITING E-BIDDING ON EPADS \nSealed tenders based on MRS – 2025 (1st Bi – Annual) / Non MRS – items for the below mentioned works are \nhereby invited on the basis of SINGLE STAGE TWO ENVELOPE PROCEDURE from the government contractors \nregistered with PEC and enlisted with C&W Department as per Centralized Enlistment Policy of the Government \nof Khyber Pakhtunkhwa for current financial year as per following schedule: \n \nS.No Work Name PEC \nCategory \nEstimated \nCost (M) \nBid \nSecurity \n@ 2% (M) \nLast date & \ntime of \nsubmission

In [17]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200): #chunk_overlap: some number of text may get overlap with the next chunk
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [18]:
chunks=split_documents(all_pdf_documents)
chunks
# Chunks are done on the basis of chunk_size and chunk_overlap
# Splitting/chunking is done

Split 40 documents into 17 chunks

Example chunk:
Content: OFFICE OF THE ASSISTANT DIRECTOR LOCAL GOVERNMENT AND RURAL 
DEVELOPMENT DEPARTMENT MANSEHRA 
NOTICE INVITING E-BIDDING ON EPADS 
Sealed tenders based on MRS – 2025 (1st Bi – Annual) / Non MRS – items...
Metadata: {'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-03-24T13:22:45+05:00', 'author': 'Dell', 'moddate': '2026-03-26T14:52:52+05:00', 'source': '..\\data\\pdf\\EE Tender.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'EE Tender.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-03-24T13:22:45+05:00', 'author': 'Dell', 'moddate': '2026-03-26T14:52:52+05:00', 'source': '..\\data\\pdf\\EE Tender.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'EE Tender.pdf', 'file_type': 'pdf'}, page_content='OFFICE OF THE ASSISTANT DIRECTOR LOCAL GOVERNMENT AND RURAL \nDEVELOPMENT DEPARTMENT MANSEHRA \nNOTICE INVITING E-BIDDING ON EPADS \nSealed tenders based on MRS – 2025 (1st Bi – Annual) / Non MRS – items for the below mentioned works are \nhereby invited on the basis of SINGLE STAGE TWO ENVELOPE PROCEDURE from the government contractors \nregistered with PEC and enlisted with C&W Department as per Centralized Enlistment Policy of the Government \nof Khyber Pakhtunkhwa for current financial year as per following schedule: \n \nS.No Work Name PEC \nCategory \nEstimated \nCost (M) \nBid \nSecurity \n@ 2% (M) \nLast date & \ntime of \nsubmission

### Embedding and VectorDB

In [19]:
import numpy as np
from sentence_transformers import SentenceTransformer # embedding model in this
import chromadb
from chromadb.config import Settings
import uuid # every record has a unique id in Vector DB
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity #retrivial from vector db

d:\ai-learning-roadmap\complete_agentic_ai\trag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"): #hugging face model trandforming text into embeddings 
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self): # _ means this is protected func
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}") # every text will be converted into 384 dimensions
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray: # returns numpyarray
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## it the embedding manager
embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1228.76it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\Sanaullah\AppData\Local\Temp\ipykernel_15124\3300585592.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}") # every text will be converted into 384 dimensions


### VectorStore

In [21]:
# class VectorStore:
#     """Manages document embeddings in a ChromaDB vector store"""
    
#     def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"): # persist mean we are storing it in hardware
#         """
#         Initialize the vector store
        
#         Args:
#             collection_name: Name of the ChromaDB collection
#             persist_directory: Directory to persist the vector store
#         """
#         self.collection_name = collection_name
#         self.persist_directory = persist_directory
#         self.client = None
#         self.collection = None
#         self._initialize_store() # this will initialize the vector store    

#     def _initialize_store(self):
#         """Initialize ChromaDB client and collection"""
#         try:
#             # Create persistent ChromaDB client
#             os.makedirs(self.persist_directory, exist_ok=True)
#             self.client = chromadb.PersistentClient(path=self.persist_directory)
#             # it is going to create a client which will have a reference to chroma vector db

#             # Get or create collection, where we are going to store collection isnide our vector db: in collection
#             self.collection = self.client.get_or_create_collection(
#                 name=self.collection_name,
#                 metadata={
#                     "hnsw:space": "cosine",  # Explicitly use cosine distance //// added
#                     "description": "PDF document embeddings for RAG"}
#             )
#             print(f"Vector store initialized. Collection: {self.collection_name}")
#             print(f"Existing documents in collection: {self.collection.count()}")
            
#         except Exception as e:
#             print(f"Error initializing vector store: {e}")
#             raise

#     def add_documents(self, documents: List[Any], embeddings: np.ndarray): # numpy array
#         """
#         Add documents and their embeddings to the vector store
        
#         Args:
#             documents: List of LangChain documents
#             embeddings: Corresponding embeddings for the documents
#         """
#         if len(documents) != len(embeddings):
#             raise ValueError("Number of documents must match number of embeddings")
        
#         print(f"Adding {len(documents)} documents to vector store...")
        
#         # Prepare data for ChromaDB
#         ids = []
#         metadatas = []
#         documents_text = []
#         embeddings_list = []
        
#         for i, (doc, embedding) in enumerate(zip(documents, embeddings)): # zip: creating tuple with embeddings, and embedding is coming from previos func
#             # Generate unique ID
#             doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}" # uuid is id for specific record
#             ids.append(doc_id)
            
#             # Prepare metadata to put it inside vector db, we are iterating through all docs
#             metadata = dict(doc.metadata)
#             metadata['doc_index'] = i
#             metadata['content_length'] = len(doc.page_content)
#             metadatas.append(metadata)
            
#             # Document content
#             documents_text.append(doc.page_content)
            
#             # Embedding
#             embeddings_list.append(embedding.tolist())
        
#         # Add to collection
#         try:
#             self.collection.add(
#                 ids=ids,
#                 embeddings=embeddings_list,
#                 metadatas=metadatas,
#                 documents=documents_text
#             )
#             print(f"Successfully added {len(documents)} documents to vector store")
#             print(f"Total documents in collection: {self.collection.count()}")
            
#         except Exception as e:
#             print(f"Error adding documents to vector store: {e}")
#             raise

# vectorstore=VectorStore() # initialize vector store
# vectorstore







class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Set hnsw:space to "cosine" so distance maps cleanly to similarity
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "hnsw:space": "cosine", 
                    "description": "PDF document embeddings for RAG"
                }
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        ids, metadatas, documents_text, embeddings_list = [], [], [], []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
        
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents: {e}")
            raise

vectorstore=VectorStore() # initialize vector store
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [22]:
chunks

[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-03-24T13:22:45+05:00', 'author': 'Dell', 'moddate': '2026-03-26T14:52:52+05:00', 'source': '..\\data\\pdf\\EE Tender.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'EE Tender.pdf', 'file_type': 'pdf'}, page_content='OFFICE OF THE ASSISTANT DIRECTOR LOCAL GOVERNMENT AND RURAL \nDEVELOPMENT DEPARTMENT MANSEHRA \nNOTICE INVITING E-BIDDING ON EPADS \nSealed tenders based on MRS – 2025 (1st Bi – Annual) / Non MRS – items for the below mentioned works are \nhereby invited on the basis of SINGLE STAGE TWO ENVELOPE PROCEDURE from the government contractors \nregistered with PEC and enlisted with C&W Department as per Centralized Enlistment Policy of the Government \nof Khyber Pakhtunkhwa for current financial year as per following schedule: \n \nS.No Work Name PEC \nCategory \nEstimated \nCost (M) \nBid \nSecurity \n@ 2% (M) \nLast date & \ntime of \nsubmission

In [23]:
### Extracing chunks from text and Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 17 texts...


Batches: 100%|██████████| 1/1 [00:02<00:00,  2.08s/it]


Generated embeddings with shape: (17, 384)
Successfully added 17 documents to vector store
Total documents in collection: 17


### Retriever pipline from VectorStore
Rag retriveal to get the context

In [24]:
# retiever is built on top of vector store and is kind of interface connected to a vectore store, base on what ever  query we get , this retrieves gives the response back    
# class RAGRetriever:
#     """Handles query based retrieval from vector store"""
#     def __init__(self, vectorstore: VectorStore, embedding_manager: EmbeddingManager):

#         """Initialize the RAG Retriever
#         Args:
#             vectorstore: VectorStore instance containing document embeddings
#             embedding_manager: EmbeddingManager for generating query embeddings
#         """
#         self.vectorstore = vectorstore
#         self.embedding_manager = embedding_manager



#     def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
#         """
#         Retrieve relevant documents for a query
        
#         Args:
#             query: The search query
#             top_k: Number of top results to return
#             score_threshold: Minimum similarity score threshold
            
#         Returns:
#             List of dictionaries containing retrieved documents and metadata
#         """
#         print(f"Retrieving documents for query: '{query}'")
#         print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
#         # Generate query embedding, to see similarity 
#         query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
#         # Search in vector store
#         try:
#             results = self.vectorstore.collection.query(    # this is going to hit a vector db
#                 query_embeddings=[query_embedding.tolist()], # in the form of list
#                 n_results=top_k
#             )
            
#             # Process results
#             retrieved_docs = []
            
#             if results['documents'] and results['documents'][0]:   # results have a document as query
#                 documents = results['documents'][0]
#                 metadatas = results['metadatas'][0]
#                 distances = results['distances'][0]
#                 ids = results['ids'][0]
                
#                 for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
#                     # Convert distance to similarity score (ChromaDB uses cosine distance)
#                     similarity_score = 1 - distance
                    
#                     if similarity_score >= score_threshold:
#                         retrieved_docs.append({
#                             'id': doc_id,
#                             'content': document,
#                             'metadata': metadata,
#                             'similarity_score': similarity_score,
#                             'distance': distance,
#                             'rank': i + 1
#                         })
#                 print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
#             else:
#                 print("No documents found")
            
#             return retrieved_docs
            
#         except Exception as e:
#             print(f"Error during retrieval: {e}")
#             return []

# rag_retriever=RAGRetriever(vectorstore,embedding_manager)






class RAGRetriever:
    """Handles query based retrieval from vector store"""
    def __init__(self, vectorstore: VectorStore, embedding_manager: EmbeddingManager):
        self.vectorstore = vectorstore
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = -1.0) -> List[Dict[str, Any]]:
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        try:
            results = self.vectorstore.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Cosine similarity conversion
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                # FIXED: Moved print statement OUTSIDE the loop
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

                

In [25]:
rag_retriever

In [26]:
rag_retriever.retrieve("What is trizetto?")

Retrieving documents for query: 'What is trizetto?'
Top K: 5, Score threshold: -1.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 10.16it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_02f6a5a5_12',
  'content': 'Khana Pul to Nilore     FR-08B\nGulber Green to T-Chowk vis Isb-Highway    FR-15\nTaxila to Hassan Abdal via N5    FR-12\nGolara Morh Station to Fateh Jang via SH    FR-13\nPIMS Hospital to Golara Shareef via G9, F10 & F11    FR-06\nGolara Mor to Khanna Pul via IJP Road    FR-09\nGolara Morh to Taxila via Margalla Road & GT Road     FR-05\nPIMS Hospital to Bari Amam to QAU    FR-04\nPIMS Hospital to NL via Margalla Road & Pak Sec.    FR-03B\nGolara Morh to Taxila via N5    FR-10\nSohan to AOU via I8    FR-02\nPIMS to NUST/PF via G-8, G9, G10 & G11    FR-07\nKhana Pul to NUST via IH & I-9, I-10 & I-11    FR-01\nGolara Morh Station to I-16 via SH & N5    FR-11\nOrange Line Metrobus Service\nRed Line Metrobus Service\nRawal DamChowk\nKashmirChowk\nDhokri\nBari ImamMuhallah Noori BaghCommunity Health centreQuaid e Azam ColonyC-Type Quaid e Azam ColonyBabul QuaidQuaid e Azam universityFR-04\nRanger’s Gate\nMuslim Colony\nAwan e Sadar\nDiplomatic Encl

### Integrate  VectorDB contex pipeline with LLM output

In [49]:
#simple RAG pipeline with groq llm
import os
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

llm = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model_name="openai/gpt-oss-120b",
    temperature=0.1,
    max_tokens=512
)


In [52]:
# Simple RAG function with retrieve context + generate response
def simple_rag(query, retriever, llm, top_k=3) -> str:
    # retrieve context
    results = retriever.retrieve(query, top_k=top_k)
    context="\n\n".join([doc['content'][:500] for doc in results]) if results else "" #whatever content we are getting, join with double new line, if they are empty  we are keeping them emty
    if not context:
        print("No context found")
    context = context[:2000]  # keep first 2000 characters

    # generate response
    prompt = """Use the following context to answer the question accurately and concisely.
    Context: {context}
    Question: {query}
    Answer:"""
    response = llm.invoke(prompt.format(context=context, query=query))

    final_prompt = prompt.format(context=context, query=query)
    print("Prompt length (chars):", len(final_prompt))
    print("Prompt length (tokens approx):", len(final_prompt.split()))
    
    return response.content


    

In [56]:
answer = simple_rag("summary of trizetto email thread?", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'summary of trizetto email thread?'
Top K: 3, Score threshold: -1.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.51it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Prompt length (chars): 1652
Prompt length (tokens approx): 251
I’m sorry, but the information you provided does not include any details about a Trizetto email thread, so I’m unable to generate a summary of it. If you can share the relevant email content or key points, I’d be happy to help summarize it for you.


### Enhanced RAG Pipeline Features

In [ ]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=3, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results]) # coming from vector db
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Hard Negative Mining Technqiues", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Hard Negative Mining Technqiues'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 10.70it/s]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Answer: **Hard‑negative mining** is a strategy used in training classifiers (especially deep‑learning models for detection, segmentation, metric learning, etc.) to focus the learning on the most confusing or “hard” negative examples—those that the current model mistakenly classifies as positive or that produce a high loss. By emphasizing these samples, the model learns a more discriminative decision boundary and converges faster.

Below is a concise summary of the most common hard‑negative‑mining techniques:

| Technique | How it works | Typical use‑cases | Pros | Cons |
|-----------|--------------|-------------------|------|------|
| **Offline (pre‑computed) mining** | After an initial pass, rank all negative samples by loss or confidence and select the top‑k hardest for the next training epoch. | Object detection (e.g., SVM‑based detectors), metric‑learning with triplet loss. | Simple to implement; dete

### Advanced RAG Pipeline: Streaming, Citations, History, Summarization

In [58]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what required for this tender?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what required for this tender?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.68it/s]

Generated embeddings with shape: (1, 384)


Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
14. Financial bids will be opened on latter and date will be communicated to technically qualified bidders only.  
15. The following documents are required to be submitted with the technical bid. Failure to which the bid will be 
considered Non-Responsive and will not be technically evaluated further:

OFFICE OF THE ASSISTANT DIRECTOR LOCAL GOVERNMENT AND RURAL 
DEVELOPMENT DEPARTMENT MANSEHRA 
NOTICE INVITING E-BIDDING ON EPADS 
Sealed tenders based on MRS – 2025 (1st Bi – Annual) / Non MRS – items for the below mentioned works are 
hereby invited on the basis of SINGLE STAGE TWO ENVELOPE PROCEDURE from the government contractors 
registered with PEC and enlisted with C&W Department as per Centralized Enlistment Policy of the Government 
of Khyber Pakhtunkhwa for current financial year as per following schedule: 
 
S.No Work Name PEC 
Category 
Estimated 
Cost